In [1]:
# 1. Load data (same way pipeline does)
import duckdb
from config import DB_PATH

con = duckdb.connect(DB_PATH)

team_daily_stats = con.execute("""
    SELECT *
    FROM team_daily_stats_backup
    WHERE season_id = 20252026
""").df()




In [11]:
con.close()
print("DuckDB connection closed.")

DuckDB connection closed.


In [2]:

# 2. Build features
from feature_engineering import build_features

df_features = build_features(team_daily_stats.copy())





In [3]:

# 3. Validate feature contract
import joblib
from config import MODEL_PATH

bundle = joblib.load(MODEL_PATH)
FEATURE_COLS = bundle["features"]

missing_features = [c for c in FEATURE_COLS if c not in df_features.columns]

print("Missing:", missing_features)

Missing: []


In [6]:
from prediction_model.inference.inference import score_dataframe

preds = score_dataframe(df_features.head(100))
preds.head()

ℹ️ Ignored 182 extra features not used by model


,season_id,game_id,game_date,team_id,tri_code,team_name,game_win,wins_before,games_before,win_pct_before,...,goal_diff_per_game,last10_points_pct,momentum,rank_pressure,urgency,momentum_x_season,urgency_x_points,playoff_prob_xgb,playoff_prob_rf,playoff_prob
0,20252026,2025020001,2025-10-07,13,FLA,Florida Panthers,1,0,0,NaN,...,0.999999,1.0,-4.999998e-07,-6,-5.926829,-6.097558e-09,-5.926829,0.618889,0.519658,0.569273
1,20252026,2025020001,2025-10-07,16,CHI,Chicago Blackhawks,0,0,0,NaN,...,-0.999999,0.0,0.000000e+00,7,6.914634,0.000000e+00,0.000000,0.170037,0.229788,0.199913
2,20252026,2025020002,2025-10-07,3,NYR,New York Rangers,0,0,0,NaN,...,-2.999997,0.0,0.000000e+00,8,7.902439,0.000000e+00,0.000000,0.261127,0.298352,0.279739
3,20252026,2025020002,2025-10-07,5,PIT,Pittsburgh Penguins,1,0,0,NaN,...,2.999997,1.0,-4.999998e-07,-7,-6.914634,-6.097558e-09,-6.914634,0.442692,0.357334,0.400013
4,20252026,2025020003,2025-10-07,21,COL,Colorado Avalanche,1,0,0,NaN,...,2.999997,1.0,-4.999998e-07,-7,-6.914634,-6.097558e-09,-6.914634,0.353597,0.540491,0.447044


In [8]:
con.execute("DESCRIBE playoff_daily_predictions").df()




,column_name,column_type,null,key,default,extra
0,season_id,BIGINT,NO,PRI,None,None
1,game_date,DATE,NO,PRI,None,None
2,team_id,BIGINT,NO,PRI,None,None
3,tri_code,VARCHAR,YES,None,None,None
4,conference,VARCHAR,YES,None,None,None
5,division,VARCHAR,YES,None,None,None
6,conference_rank,INTEGER,YES,None,None,None
7,division_rank,INTEGER,YES,None,None,None
8,wildcard_rank,INTEGER,YES,None,None,None
9,points,INTEGER,YES,None,None,None


In [9]:

con.execute("DESCRIBE playoff_shap_values").df()

,column_name,column_type,null,key,default,extra
0,win_pct_before,FLOAT,YES,None,None,None
1,sos_past,FLOAT,YES,None,None,None
2,sos_future,FLOAT,YES,None,None,None
3,pp_net_pct,FLOAT,YES,None,None,None
4,times_sh,FLOAT,YES,None,None,None
...,...,...,...,...,...,...
58,team_id,BIGINT,YES,None,None,None
59,tri_code,VARCHAR,YES,None,None,None
60,season_id,BIGINT,YES,None,None,None
61,game_date,TIMESTAMP,YES,None,None,None


In [10]:
con.execute("""
SELECT *
FROM playoff_daily_predictions
WHERE game_date = '2026-03-01'
""").df()

,season_id,game_date,team_id,tri_code,conference,division,conference_rank,division_rank,wildcard_rank,points,projected_points,playoff_prob,playoff_prob_xgb,playoff_prob_rf,playoff_prob_display,playoff_tier,season_bucket,model_name,model_version,model_run_ts
0,20252026,2026-03-01,1,NJD,Eastern,Metropolitan,15,7,9,60,82.0,0.543804,0.573575,0.514033,0.543804,Bubble,late,playoff_models_xgb_rf,wc_field_pressure_v4,2026-03-15 14:46:20.565514
1,20252026,2026-03-01,2,NYI,Eastern,Metropolitan,6,3,0,75,101.0,0.643091,0.753830,0.532353,0.643091,Bubble,late,playoff_models_xgb_rf,wc_field_pressure_v4,2026-03-15 14:46:20.565514
2,20252026,2026-03-01,3,NYR,Eastern,Metropolitan,16,8,10,53,74.0,0.539693,0.564502,0.514884,0.539693,Bubble,late,playoff_models_xgb_rf,wc_field_pressure_v4,2026-03-15 14:46:20.565514
3,20252026,2026-03-01,4,PHI,Eastern,Metropolitan,12,6,6,65,90.0,0.576895,0.619552,0.534239,0.576895,Bubble,late,playoff_models_xgb_rf,wc_field_pressure_v4,2026-03-15 14:46:20.565514
4,20252026,2026-03-01,5,PIT,Eastern,Metropolitan,4,2,0,75,104.0,0.647880,0.756413,0.539347,0.647880,Bubble,late,playoff_models_xgb_rf,wc_field_pressure_v4,2026-03-15 14:46:20.565514
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,20252026,2026-03-01,30,MIN,Western,Central,3,3,0,80,108.0,0.592333,0.730382,0.454283,0.592333,Bubble,late,playoff_models,v5,2026-03-22 15:35:10.083403
60,20252026,2026-03-01,52,WPG,Western,Central,12,6,6,56,78.0,0.516712,0.615594,0.417830,0.516712,Bubble,late,playoff_models,v5,2026-03-22 15:35:10.083403
61,20252026,2026-03-01,54,VGK,Western,Pacific,4,1,0,70,96.0,0.591455,0.724795,0.458115,0.591455,Bubble,late,playoff_models,v5,2026-03-22 15:35:10.083403
62,20252026,2026-03-01,55,SEA,Western,Pacific,8,4,2,65,90.0,0.533536,0.660721,0.406352,0.533536,Bubble,late,playoff_models,v5,2026-03-22 15:35:10.083403
